# Setup del entorno de trabajo

Notebook de infraestructura. **No forma parte del pipeline** — por eso no lleva número: los
notebooks numerados son los entregables que se ejecutan en orden, este corre una sola vez al
montar el proyecto.

Hace dos cosas:

1. **Verifica la conexión a Lakebase** de extremo a extremo, antes de que nada dependa de ella.
2. **Crea el catálogo de trabajo** en Unity Catalog con sus tres esquemas.

---

### Sin credenciales almacenadas

Lakebase autentica con tokens OAuth de 60 minutos generados en el momento de conectar a partir
de la identidad del workspace. No hay contraseña que guardar en un secret scope.

### Por qué un catálogo propio

Se podría trabajar sobre el catálogo `workspace` que viene por defecto, pero un catálogo por
proyecto da aislamiento real: permisos independientes, linaje legible, y un borrado de una línea
si hay que empezar de cero. Además el nombre coincide con el esquema de Postgres, lo que hace que
el diagrama de arquitectura se lea solo — `bank_churn` operacional en Lakebase, `bank_churn`
analítico en Unity Catalog.

## 1. Entorno

El SDK 0.67 no incluye `w.postgres`. Hay que actualizarlo y después reiniciar el proceso de Python.


In [0]:
# pg8000 en lugar de psycopg: es un driver de Postgres escrito en Python puro, así
# que no lleva ninguna libpq compilada que pueda tumbar el kernel serverless.
# protobuf fijado por debajo de la 6.x: al actualizar el SDK entra protobuf 6, que
# entra en conflicto con grpcio-status y googleapis-common-protos y puede romper Spark.
%pip install --quiet --upgrade "databricks-sdk>=0.81.0" "pg8000" "protobuf>=5.26.1,<6"
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import importlib.metadata

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

print("versión del SDK :", importlib.metadata.version("databricks-sdk"))
print("protobuf        :", importlib.metadata.version("protobuf"))
print("pg8000          :", importlib.metadata.version("pg8000"))
print("tiene w.postgres:", hasattr(w, "postgres"))

SDK version : 0.123.0
protobuf    : 5.29.6
pg8000      : 1.31.5
has w.postgres: True


## 2. Explorar la superficie de la API

En lugar de adivinar los nombres de los métodos, imprimimos lo que el SDK instalado expone de
verdad. Este paso existe porque ya nos equivocamos una vez: buscabamos
`w.database.generate_database_credential`, que no existe, cuando lo correcto es `w.postgres`.


In [0]:
import inspect

api = w.postgres
print("Métodos disponibles en w.postgres:\n")
for name in sorted(n for n in dir(api) if not n.startswith("_")):
    attr = getattr(api, name)
    if callable(attr):
        try:
            print(f"  {name}{inspect.signature(attr)}")
        except (ValueError, TypeError):
            print(f"  {name}(...)")

Methods on w.postgres:

  create_branch(parent: 'str', branch: 'Branch', branch_id: 'str', *, replace_existing: 'Optional[bool]' = None) -> 'CreateBranchOperation'
  create_catalog(catalog: 'Catalog', catalog_id: 'str') -> 'CreateCatalogOperation'
  create_cdf_config(parent: 'str', cdf_config: 'CdfConfig', *, cdf_config_id: 'Optional[str]' = None) -> 'CreateCdfConfigOperation'
  create_data_api(parent: 'str', data_api: 'DataApi') -> 'CreateDataApiOperation'
  create_database(parent: 'str', database: 'Database', *, database_id: 'Optional[str]' = None, replace_existing: 'Optional[bool]' = None) -> 'CreateDatabaseOperation'
  create_endpoint(parent: 'str', endpoint: 'Endpoint', endpoint_id: 'str', *, replace_existing: 'Optional[bool]' = None) -> 'CreateEndpointOperation'
  create_project(project: 'Project', project_id: 'str') -> 'CreateProjectOperation'
  create_replication_group_preview(parent: 'str', replication_group_preview: 'ReplicationGroupPreview', replication_group_preview_id: 'st

## 3. Localizar el proyecto, la rama y el punto de conexión

Al crear un proyecto se aprovisionan automáticamente una rama `production` y un punto de conexión
`primary` de lectura y escritura. Aun así los resolvemos preguntandoselo a la API en vez de
escribir los nombres a mano: si mañana cambian, el notebook sigue funcionando.


In [0]:
PROJECT_ID = "bank-churn-prediction"
PROJECT    = f"projects/{PROJECT_ID}"

branches = list(w.postgres.list_branches(PROJECT))
for b in branches:
    print("rama  :", b.name)

BRANCH = branches[0].name          # projects/<id>/branches/<id>
print("\nrama en uso:", BRANCH)

endpoints = list(w.postgres.list_endpoints(BRANCH))
for e in endpoints:
    print("punto de conexión:", e.name)

ENDPOINT = endpoints[0].name       # projects/<id>/branches/<id>/endpoints/<id>
print("\npunto de conexión en uso:", ENDPOINT)

branch  : projects/bank-churn-prediction/branches/production

using branch: projects/bank-churn-prediction/branches/production
endpoint: projects/bank-churn-prediction/branches/production/endpoints/primary

using endpoint: projects/bank-churn-prediction/branches/production/endpoints/primary


In [0]:
# El host se resuelve desde el punto de conexión, no desde el valor copiado de la interfaz
ep = w.postgres.get_endpoint(ENDPOINT)

PGHOST     = ep.status.hosts.host
PGDATABASE = "databricks_postgres"
PGPORT     = 5432
PGSSLMODE  = "require"
SCHEMA     = "bank_churn"

# Identidad del workspace: el rol de Postgres que se creo junto con el proyecto
PGUSER = w.current_user.me().user_name

print("host          :", PGHOST)
print("base de datos :", PGDATABASE)
print("usuario       :", PGUSER)

host    : ep-calm-art-d80j4anc.database.us-east-2.cloud.databricks.com
database: databricks_postgres
user    : <usuario>


## 4. Generar una credencial de vida corta

La credencial queda acotada a la ruta del punto de conexión y vale 60 minutos. La caducidad se
comprueba al iniciar sesión, así que una consulta que ya está corriendo no se interrumpe a mitad.

Esto es lo que hace que no haga falta guardar ninguna contraseña en ningún sitio.


In [0]:
def get_token() -> str:
    """Devuelve una credencial OAuth de Lakebase recien generada, valida 60 minutos."""
    cred = w.postgres.generate_database_credential(ENDPOINT)
    return cred.token


token = get_token()

# El token no se imprime nunca: una salida guardada en el notebook y subida a GitHub
# sería una fuga real de credenciales.
print("credencial obtenida, longitud:", len(token))

token acquired — length: 835


## 5. Conectar y verificar el esquema

`pg8000` es un driver escrito en Python puro. Al no llevar `libpq` compilada, no puede tumbar el
kernel como hacia `psycopg[binary]`. Habla el protocolo de Postgres directamente.

Es mas lento que psycopg en transferencias grandes, y aquí da igual: esta conexión lleva DDL y una
inserción de 10.000 filas, no lecturas analíticas. Las lecturas masivas van por Spark JDBC, mas
abajo.


In [0]:
import ssl

import pg8000.dbapi

ctx = ssl.create_default_context()          # equivale a sslmode=require

conn = pg8000.dbapi.connect(
    host=PGHOST,
    port=PGPORT,
    database=PGDATABASE,
    user=PGUSER,
    password=token,
    ssl_context=ctx,
)

try:
    cur = conn.cursor()

    cur.execute("SELECT version();")
    print(cur.fetchone()[0], "\n")

    cur.execute(
        """
        SELECT table_name, table_type
        FROM   information_schema.tables
        WHERE  table_schema = %s
        ORDER  BY table_type, table_name;
        """,
        (SCHEMA,),
    )
    print(f"Objetos en el esquema {SCHEMA}:")
    for name, kind in cur.fetchall():
        print(f"  {kind:<10} {name}")

    cur.execute(f"SELECT * FROM {SCHEMA}.geographies ORDER BY geography_id;")
    print("\ngeographies:")
    for row in cur.fetchall():
        print("  ", row)
finally:
    conn.close()

PostgreSQL 17.10 (4f20678) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit 

Objects in schema bank_churn:
  BASE TABLE customers_raw
  BASE TABLE geographies

geographies:
   [1, 'France', 'FR']
   [2, 'Spain', 'ES']
   [3, 'Germany', 'DE']


## 6. El mismo camino, ahora por Spark JDBC

Se nos pidió expresamente una conexión JDBC, así que la hacemos: misma credencial, pasada como
contraseña de JDBC. Este es el camino de lectura que después llenara la tabla Bronze.

Si al classpath del entorno serverless le falta el driver de Postgres, esta celda falla. Eso no
es un bloqueo sino información útil: leer con `pg8000` y pasarle las filas a
`spark.createDataFrame` es una alternativa perfectamente valida para 10.000 filas.


In [0]:
jdbc_url = f"jdbc:postgresql://{PGHOST}:{PGPORT}/{PGDATABASE}?sslmode={PGSSLMODE}"

try:
    df = (
        spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("driver", "org.postgresql.Driver")
        .option("dbtable", f"{SCHEMA}.geographies")
        .option("user", PGUSER)
        .option("password", token)
        .load()
    )
    df.show()
    print("filas leídas por JDBC:", df.count())
except Exception as exc:
    print(f"camino JDBC no disponible -> {type(exc).__name__}: {exc}")
    print("Alternativa: leer con pg8000 y pasar las filas a spark.createDataFrame.")

+------------+------------+--------+
|geography_id|country_name|iso_code|
+------------+------------+--------+
|           1|      France|      FR|
|           2|       Spain|      ES|
|           3|     Germany|      DE|
+------------+------------+--------+

rows read via JDBC: 3


## 7. Catálogo de trabajo

Tres esquemas siguiendo la arquitectura medallion. Los `COMMENT` no son decorativos: aparecen en
el explorador de Unity Catalog y son la primera documentación que ve quien llegue al proyecto sin
contexto.

In [ ]:
CATALOG = "bank_churn"

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS {CATALOG}
    COMMENT 'Predicción de abandono de clientes bancarios - proyecto integrador'
""")

ESQUEMAS = {
    "bronze": "Copia inmutable de la extracción JDBC desde Lakebase. Sin transformaciones.",
    "silver": "Datos tipados, limpios y con variables derivadas deterministas. Fuente para el modelado.",
    "gold":   "Predicciones, métricas y tablas listas para que las consuma la aplicación.",
}

for nombre, descripcion in ESQUEMAS.items():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{nombre} COMMENT '{descripcion}'")
    print(f"  {CATALOG}.{nombre}")

print("\ncatalogo listo")

In [ ]:
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))

# El catalogo por defecto debe quedar limpio: solo 'default'
print("contenido de workspace (debería quedar solo default):")
display(spark.sql("SHOW SCHEMAS IN workspace"))

## Resultado

Entorno listo:

```
Lakebase                          Unity Catalog
  bank_churn.customers_raw   ──▶    bank_churn.bronze
  bank_churn.customers                bank_churn.silver
  bank_churn.customer_predictions     bank_churn.gold
```

Conexión verificada por **pg8000** y por **Spark JDBC**, y catálogo analítico creado con sus tres
esquemas.

**Siguiente:** `01_ingesta_bd_nube` — descarga desde Kaggle con trazabilidad, verificaciones de
calidad, carga a Lakebase y primera extracción JDBC hacia Bronze.